In [1]:
from pyspark.sql import SparkSession

In [10]:
from pyspark.sql import SparkSession

# Initialize local Spark Session with security configurations
spark = SparkSession.builder \
    .appName("NYCtaxiDataAnalysisproject") \
    .config("spark.driver.memory", "4g") \
    .config("spark.hadoop.security.authentication", "simple") \
    .config("spark.hadoop.security.authorization", "false") \
    .getOrCreate()


26/06/10 18:35:06 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, LongType

# Define explicit schema based on the NYC TLC data dictionary
yellow_taxi_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    # Note: passenger_count and RatecodeID are semantically integers, but frequently
    # stored as Double/Float in TLC parquet files due to upstream NaN handling.
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True)  # Enacted Jan 2025
])

In [15]:
# Install required packages if needed
import subprocess
import sys

for package in ["pyarrow", "pandas", "numpy"]:
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import pyarrow.parquet as pq
import numpy as np

# Read the file with PyArrow (no Hadoop/Spark issues)
print("Loading parquet file with PyArrow...")
table = pq.read_table("/Users/avikumart/Documents/GitHub/DA-DS-Questions/PySpark/yellow_tripdata_2026-04.parquet")

# Display schema
print("\nParquet Schema:")
print(table.schema)

# Convert directly to Spark DataFrame using PyArrow Table
# This bypasses pandas entirely
print("\nConverting to Spark DataFrame...")
df = spark.createDataFrame(table.to_pylist())

# Verify the result
print(f"✓ Successfully loaded {df.count()} rows")
print("\nDataFrame Schema:")
df.printSchema()

# Show sample data
print("\nFirst 5 rows:")
df.show(5)


Loading parquet file with PyArrow...

Parquet Schema:
VendorID: int32
tpep_pickup_datetime: timestamp[us]
tpep_dropoff_datetime: timestamp[us]
passenger_count: int64
trip_distance: double
RatecodeID: int64
store_and_fwd_flag: large_string
PULocationID: int32
DOLocationID: int32
payment_type: int64
fare_amount: double
extra: double
mta_tax: double
tip_amount: double
tolls_amount: double
improvement_surcharge: double
total_amount: double
congestion_surcharge: double
Airport_fee: double
cbd_congestion_fee: double

Converting to Spark DataFrame...


26/06/10 18:43:45 WARN TaskSetManager: Stage 0 contains a task of very large size (62651 KiB). The maximum recommended task size is 1000 KiB.


✓ Successfully loaded 3831240 rows

DataFrame Schema:
root
 |-- Airport_fee: double (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- VendorID: long (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)


First 5 rows:
+-----------+----------

26/06/10 18:43:48 WARN TaskSetManager: Stage 3 contains a task of very large size (62651 KiB). The maximum recommended task size is 1000 KiB.
Traceback (most recent call last):
  File "/Users/avikumart/Documents/GitHub/DA-DS-Questions/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/avikumart/Documents/GitHub/DA-DS-Questions/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


In [16]:
# write a groupy query to find the average fare amount by payment type
average_fare_by_payment_type = df.groupBy("payment_type").avg("fare_amount")
average_fare_by_payment_type.show()

26/06/10 18:45:16 WARN TaskSetManager: Stage 4 contains a task of very large size (62651 KiB). The maximum recommended task size is 1000 KiB.


+------------+------------------+
|payment_type|  avg(fare_amount)|
+------------+------------------+
|           1|20.050289348491678|
|           3| 10.50504324966079|
|           2|19.415355891355926|
|           4|  3.02163714111179|
|           0|25.917277946858484|
+------------+------------------+



In [17]:
# find the average fare amount by payment type and passenger count
average_fare_by_payment_and_passenger = df.groupBy("payment_type", "passenger_count").avg("fare_amount")
average_fare_by_payment_and_passenger.show()

26/06/10 18:45:51 WARN TaskSetManager: Stage 7 contains a task of very large size (62651 KiB). The maximum recommended task size is 1000 KiB.


+------------+---------------+------------------+
|payment_type|passenger_count|  avg(fare_amount)|
+------------+---------------+------------------+
|           3|              0|19.920735294117645|
|           3|              5|1.6600000000000001|
|           4|              0|13.777283950617285|
|           2|              6|19.296347826086954|
|           1|              3|22.637795012035525|
|           2|              5|17.305137614678905|
|           1|              0|  16.9709006368139|
|           1|              1|19.540116944653825|
|           3|              4| 7.942898550724639|
|           1|              4| 28.27103728248232|
|           4|              5| 6.746153846153846|
|           4|              3|1.3346820809248556|
|           4|              4|1.3983959537572253|
|           3|              3| 7.151322751322752|
|           4|              2|1.0867285464098073|
|           1|              5| 17.16063845153836|
|           2|              3|23.498945164667052|


In [18]:
# find the average trip count by each day of the month
average_trip_count_by_day = df.groupBy("tpep_pickup_datetime").count()
average_trip_count_by_day.show()

26/06/10 18:46:12 WARN TaskSetManager: Stage 10 contains a task of very large size (62651 KiB). The maximum recommended task size is 1000 KiB.


+--------------------+-----+
|tpep_pickup_datetime|count|
+--------------------+-----+
| 2026-04-01 00:44:27|    1|
| 2026-04-01 00:41:48|    1|
| 2026-04-01 01:46:02|    1|
| 2026-04-01 02:13:24|    1|
| 2026-04-01 04:06:13|    2|
| 2026-04-01 05:50:11|    2|
| 2026-04-01 06:05:37|    2|
| 2026-04-01 06:41:59|    2|
| 2026-04-01 06:15:17|    2|
| 2026-04-01 07:22:07|    3|
| 2026-04-01 07:26:23|    1|
| 2026-04-01 08:49:14|    2|
| 2026-04-01 08:38:26|    6|
| 2026-04-01 08:55:03|    2|
| 2026-04-01 01:27:07|    1|
| 2026-04-01 03:13:16|    1|
| 2026-04-01 05:03:21|    1|
| 2026-04-01 06:30:08|    2|
| 2026-04-01 07:42:50|    3|
| 2026-04-01 07:51:35|    1|
+--------------------+-----+
only showing top 20 rows
